In [1]:
import numpy as np
import pandas as pd
import joblib as jb
import streamlit as st

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("bmw_global_sales_2018_2025.csv")

In [3]:
df.shape

(3072, 11)

In [6]:
df.head()

,Year,Month,Region,Model,Units_Sold,Avg_Price_EUR,Revenue_EUR,BEV_Share,Premium_Share,GDP_Growth,Fuel_Price_Index
0,2018,1,Europe,3 Series,7822,47482,371404204,0.011,19.12,3.5,1.0
1,2018,1,Europe,5 Series,10280,61685,634121800,0.019,19.12,3.5,1.0
2,2018,1,Europe,X3,3105,58433,181434465,0.022,19.12,3.5,1.0
3,2018,1,Europe,X5,7420,67955,504226100,0.021,19.12,3.5,1.0
4,2018,1,Europe,X7,8474,92300,782150200,0.035,19.12,3.5,1.0


In [7]:
df.dtypes

Year                  int64
Month                 int64
Region               object
Model                object
Units_Sold            int64
Avg_Price_EUR         int64
Revenue_EUR           int64
BEV_Share           float64
Premium_Share       float64
GDP_Growth          float64
Fuel_Price_Index    float64
dtype: object

In [4]:
df.isnull().sum()

Year                0
Month               0
Region              0
Model               0
Units_Sold          0
Avg_Price_EUR       0
Revenue_EUR         0
BEV_Share           0
Premium_Share       0
GDP_Growth          0
Fuel_Price_Index    0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(0)

In [8]:
x = df.drop(['Model'],axis=1)
y=df['Model']

In [12]:
numerical_cols =x.select_dtypes(include=['int64','float64']).columns.tolist()

In [13]:
categorical_cols = x.select_dtypes(include=['object']).columns.tolist()

In [14]:

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])



In [15]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [16]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [18]:
model = Pipeline(steps=[
    ('pre',preprocessor),('reg',LogisticRegression(max_iter=1000)),
])

In [19]:
model.fit(X_train,y_train)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Year', 'Month',
                                                   'Units_Sold',
                                                   'Avg_Price_EUR',
                                                   'Revenue_EUR', 'BEV_Share',
                                                   'Premium_Share',
                                                   'GDP_Growth',
                                                   'Fuel_Price_Index']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Region'])])),
                ('reg', LogisticRegression(max_iter=1000))])

In [22]:
y_pred= model.predict(X_test)
print(f'Accuracy:{accuracy_score(y_test,y_pred)*100:.2f}')
print(f'{classification_report(y_test,y_pred,zero_division=0)}')

Accuracy:93.98
              precision    recall  f1-score   support

    3 Series       1.00      1.00      1.00        83
    5 Series       0.81      0.80      0.81        75
        MINI       1.00      1.00      1.00        89
          X3       0.94      0.95      0.95        88
          X5       0.91      0.99      0.95        72
          X7       1.00      1.00      1.00        63
          i4       0.86      0.80      0.82        83
          iX       1.00      1.00      1.00        62

    accuracy                           0.94       615
   macro avg       0.94      0.94      0.94       615
weighted avg       0.94      0.94      0.94       615



In [23]:
jb.dump(model,'LogisticRegression.pkl')

['LogisticRegression.pkl']